# Lección 4: Procesamiento de datos estructurados (Spark SQL y DataFrames)  
## Proyecto: Retail Analytics Pipeline – RetailMax

## 1. Introducción

En esta lección se trabaja con datos estructurados utilizando DataFrames y Spark SQL, herramientas optimizadas para el procesamiento eficiente en Apache Spark.

A partir del dataset utilizado en las lecciones anteriores, se transforman los RDDs en DataFrames con esquemas definidos, se ejecutan consultas SQL para obtener métricas de negocio y se preparan los datos para su uso en etapas posteriores del pipeline.


In [17]:
import os
os.environ["HADOOP_HOME"] = "C:\\hadoop"

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import *

spark = SparkSession.builder \
    .appName("RetailAnalyticsPipeline") \
    .master("local[*]") \
    .getOrCreate()

In [4]:
file_path = "data/ecommerce_client_data.csv"

df = spark.read.csv(file_path, header=True, inferSchema=True)

df.show(5)
df.printSchema()

+--------------+-----------------------+-------------+----------------------+--------------+-----------------------+-----------+---------+----------+----------+-----+----------------+-------+------+-----------+-----------------+-------+-------+
|Administrative|Administrative_Duration|Informational|Informational_Duration|ProductRelated|ProductRelated_Duration|BounceRates|ExitRates|PageValues|SpecialDay|Month|OperatingSystems|Browser|Region|TrafficType|      VisitorType|Weekend|Revenue|
+--------------+-----------------------+-------------+----------------------+--------------+-----------------------+-----------+---------+----------+----------+-----+----------------+-------+------+-----------+-----------------+-------+-------+
|             0|                    0.0|            0|                   0.0|             1|                    0.0|        0.2|      0.2|       0.0|       0.0|  Feb|               1|      1|     1|          1|Returning_Visitor|  false|  false|
|             0|    

## 4. Aplicación de esquema explícito

Se define un esquema estructurado para mejorar la consistencia y el rendimiento.

In [5]:
schema = StructType([
    StructField("Administrative", IntegerType(), True),
    StructField("Administrative_Duration", DoubleType(), True),
    StructField("Informational", IntegerType(), True),
    StructField("Informational_Duration", DoubleType(), True),
    StructField("ProductRelated", IntegerType(), True),
    StructField("ProductRelated_Duration", DoubleType(), True),
    StructField("BounceRates", DoubleType(), True),
    StructField("ExitRates", DoubleType(), True),
    StructField("PageValues", DoubleType(), True),
    StructField("SpecialDay", DoubleType(), True),
    StructField("Month", StringType(), True),
    StructField("OperatingSystems", IntegerType(), True),
    StructField("Browser", IntegerType(), True),
    StructField("Region", IntegerType(), True),
    StructField("TrafficType", IntegerType(), True),
    StructField("VisitorType", StringType(), True),
    StructField("Weekend", StringType(), True),
    StructField("Revenue", StringType(), True)
])

df = spark.read.csv(file_path, header=True, schema=schema)

df.printSchema()

root
 |-- Administrative: integer (nullable = true)
 |-- Administrative_Duration: double (nullable = true)
 |-- Informational: integer (nullable = true)
 |-- Informational_Duration: double (nullable = true)
 |-- ProductRelated: integer (nullable = true)
 |-- ProductRelated_Duration: double (nullable = true)
 |-- BounceRates: double (nullable = true)
 |-- ExitRates: double (nullable = true)
 |-- PageValues: double (nullable = true)
 |-- SpecialDay: double (nullable = true)
 |-- Month: string (nullable = true)
 |-- OperatingSystems: integer (nullable = true)
 |-- Browser: integer (nullable = true)
 |-- Region: integer (nullable = true)
 |-- TrafficType: integer (nullable = true)
 |-- VisitorType: string (nullable = true)
 |-- Weekend: string (nullable = true)
 |-- Revenue: string (nullable = true)



## 5. Exploración y transformación de datos

In [6]:
# Filtrar sesiones con valor
df_filtered = df.filter(col("PageValues") > 0)

# Seleccionar columnas relevantes
df_selected = df.select("VisitorType", "PageValues", "Month")

df_selected.show(5)

+-----------------+----------+-----+
|      VisitorType|PageValues|Month|
+-----------------+----------+-----+
|Returning_Visitor|       0.0|  Feb|
|Returning_Visitor|       0.0|  Feb|
|Returning_Visitor|       0.0|  Feb|
|Returning_Visitor|       0.0|  Feb|
|Returning_Visitor|       0.0|  Feb|
+-----------------+----------+-----+
only showing top 5 rows


## 6. Creación de vista temporal para SQL

In [7]:
df.createOrReplaceTempView("ecommerce_data")

In [8]:
# PageValues promedio por tipo de visitante
query_1 = """
SELECT VisitorType, AVG(PageValues) as avg_value
FROM ecommerce_data
GROUP BY VisitorType
ORDER BY avg_value DESC
"""

spark.sql(query_1).show()

+-----------------+-----------------+
|      VisitorType|        avg_value|
+-----------------+-----------------+
|            Other|18.19181222389411|
|      New_Visitor|10.77218748275915|
|Returning_Visitor|5.006175700140938|
+-----------------+-----------------+



In [9]:
# Total de PageValues por mes
query_2 = """
SELECT Month, SUM(PageValues) as total_value
FROM ecommerce_data
GROUP BY Month
ORDER BY total_value DESC
"""

spark.sql(query_2).show()

+-----+------------------+
|Month|       total_value|
+-----+------------------+
|  Nov|21373.877339204006|
|  May|18271.814428409994|
|  Dec|11801.010926284998|
|  Mar| 7551.113003855003|
|  Oct| 4746.423531588002|
|  Sep|3385.4579065260004|
|  Aug|2571.1842568109996|
|  Jul|1773.1066731550002|
| June| 976.7346385279994|
|  Feb|      163.82674265|
+-----+------------------+



In [10]:
# Conteo de conversiones (Revenue)
query_3 = """
SELECT VisitorType, COUNT(*) as total_sessions
FROM ecommerce_data
WHERE Revenue = 'TRUE'
GROUP BY VisitorType
ORDER BY total_sessions DESC
"""

spark.sql(query_3).show()

+-----------------+--------------+
|      VisitorType|total_sessions|
+-----------------+--------------+
|Returning_Visitor|          1470|
|      New_Visitor|           422|
|            Other|            16|
+-----------------+--------------+



## 8. Agregaciones con DataFrame API

In [11]:
df.groupBy("VisitorType") \
  .agg(avg("PageValues").alias("avg_value")) \
  .orderBy("avg_value", ascending=False) \
  .show()

+-----------------+-----------------+
|      VisitorType|        avg_value|
+-----------------+-----------------+
|            Other|18.19181222389411|
|      New_Visitor|10.77218748275915|
|Returning_Visitor|5.006175700140938|
+-----------------+-----------------+



In [ ]:
# Convertir a pandas
pdf = df_agg.toPandas()

# Guardar como CSV
pdf.to_csv("data/visitor_metrics.csv", index=False)

print("Archivo guardado con pandas correctamente")

Archivo guardado con pandas correctamente
